In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


# Code Evaluation for Circuit Analysis

This notebook performs a static code evaluation of the circuit analysis implementation at `/net/scratch2/smallyan/rome_eval`.

**Evaluation Mode: No-Execution Code Evaluation**

We will:
1. Read the Plan and codewalk files to understand the project goal
2. Evaluate each code block for Runnable, Correct-Implementation, Redundant, and Irrelevant flags
3. Compute quantitative metrics
4. Produce a binary checklist summary

In [2]:
# First, let's explore the repo structure and find the Plan and codewalk files
repo_path = '/net/scratch2/smallyan/rome_eval'
for root, dirs, files in os.walk(repo_path):
    level = root.replace(repo_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        print(f'{subindent}{file}')

rome_eval/
  globals.yml
  CodeWalkthrough.md
  .gitignore
  plan.md
  CITATION.cff
  documentation.pdf
  LICENSE
  .gitattributes
  util/
    globals.py
    __init__.py
    hparams.py
    runningstats.py
    nethook.py
    generate.py
    perplexity.py
    logit_lens.py
    __pycache__/
      globals.cpython-311.pyc
      perplexity.cpython-311.pyc
      hparams.cpython-311.pyc
      __init__.cpython-311.pyc
      logit_lens.cpython-311.pyc
      runningstats.cpython-311.pyc
      generate.cpython-311.pyc
      nethook.cpython-311.pyc
  hparams/
    FT/
      EleutherAI_gpt-j-6B_unconstr.json
      EleutherAI_gpt-j-6B_constr.json
      gpt2-xl_unconstr.json
      gpt2-medium_constr.json
      gpt2-xl_attn.json
      gpt2-xl_constr.json
      gpt2-large_constr.json
    KE/
      gpt2-xl_zsRE.json
      gpt2-xl_CF.json
      gpt2-xl.json
    MEND/
      gpt2-xl_zsRE.json
      EleutherAI_gpt-j-6B_CF.json
      gpt2-xl.json
      EleutherAI_gpt-j-6B.json
      gpt2-xl_CF.json
    ROME/
 

      02/
        7cbd62ff188457e75ff52f8db1ab61b9e032d6
        e1c08f083c83c95c352ddcc1bd4f68aaae147f
      75/
        d46da3291dfda68cbe821a80fdbaf5459d02d9
      f7/
        192ced6ec49b42d9fee0738aa0783ec2b469c4
        11e7aa11b5c766369c0026bad765ca8363db73
      a0/
        6cf5397317ed56119f2d52ee098c5967304693
      08/
        119fb5a5d2c34a9e116cadca8e7b6e4db23e5a
      2c/
        7990e9914e99562fccee5270ff1faaf72ebbd3
        c48914c696c72f9e045a2ae33615dae8bccdb4
      e5/
        62787d5f385a340fe33290f928a7f056718d38
      30/
        be2b1e317c5fea37e8ab3052856000c07c88c6
      1a/
        cedd08fd5453ab6fa0558af0e57781ccf18926
      b8/
        e66f957756e68625d9b3b00ba90e178d41a951
      ee/
        fca094239878901d695e81a550d9efe57834db
      bb/
        e6914990ca96944ee679e12c0d3f80382e32ac
      11/
        31bb85c343fa380c06360e4df20ed344054a7c
        9ea3233190c409a58e25510de2e20a6702c78d
      88/
        c9140b5cce5e7cbeb01c96c3c9ed34c3356fc2
      pack/
  

        experiment/
          gen.yaml
          fc.yaml
          qa.yaml
      __pycache__/
        hooks.cpython-311.pyc
        mend_main.cpython-311.pyc
        efk_main.cpython-311.pyc
        __init__.cpython-311.pyc
        utils.cpython-311.pyc
        editable_model.cpython-311.pyc
        nn.cpython-311.pyc
        efk_hparams.cpython-311.pyc
        mend_hparams.cpython-311.pyc
        losses.cpython-311.pyc
      algs/
        enn.py
        efk.py
        mend.py
        ft.py
        __pycache__/
          mend.cpython-311.pyc
          efk.cpython-311.pyc
      data_classes/
        wiki.py
        nq.py
        fever.py
        zsre.py
  data/
    attribute_snippets.json
    idf.npy
    tfidf_vocab.json
    counterfact.json
    known_1000.json
    stats/
      gpt2/
        wikipedia_stats/
      gpt2-xl/
        wikipedia_stats/
          transformer.h.17.mlp.c_proj_float32_mom2_100000.npz
  scripts/
    setup_clean_ipynb.sh
    setup_conda.sh
    rome.yml
    causal_

## Step 1: Read Plan and Codewalk Files

Let's read the plan.md and CodeWalkthrough.md files to understand the project goal.

In [3]:
# Store plan and codewalk content
plan_content = """# Plan
## Objective
Analyze the storage and recall of factual associations in autoregressive transformer language models, finding evidence that these associations correspond to localized, directly-editable computations.

## Hypothesis
1. Factual associations in GPT correspond to a localized computation mechanism where each midlayer MLP module accepts inputs encoding a subject, then produces outputs recalling memorized properties about that subject, with middle layer MLP outputs accumulating information that is copied to the last token by attention at high layers.
2. Factual associations are localized in the MLP modules at specific middle layers, specifically at the processing of the subject's last token.
3. MLP layers in transformers can be modeled as linear associative memory where weights act as key-value stores.

## Methodology
1. Develop a causal intervention method (Causal Tracing) using causal mediation analysis to identify neuron activations that are decisive in a model's factual predictions by running the network with corrupted subject embeddings and selectively restoring hidden states.
2. Modify feed-forward weights using Rank-One Model Editing (ROME) to update specific factual associations. ROME inserts a new key-value association into a single MLP layer by computing a rank-one weight update that minimizes interference with existing memories.
3. Evaluate ROME on both a standard zero-shot relation extraction (zsRE) benchmark and a new COUNTERFACT dataset of difficult counterfactual assertions, measuring efficacy, generalization (paraphrase), and specificity (neighborhood).
4. Experimental setting: GPT-2 XL (1.5B parameters) and GPT-J (6B parameters) autoregressive transformer models tested on factual statements and counterfactual edits.

## Experiments
### Causal Tracing of Factual Associations
- What varied: Layer and token position of hidden state mediators; corruption applied to subject embeddings
- Metric: Average Indirect Effect (AIE) measuring contribution of hidden states to factual prediction restoration
- Main result: MLP modules at middle layers (around layer 15-18) at the last subject token have strong causal effects (AIE=6.6% for MLP vs 1.6% for attention at early site), revealing a distinct early site in middle-layer feed-forward modules.

### ROME Evaluation on Zero-Shot Relation Extraction (zsRE)
- What varied: Model editing method (ROME vs FT, FT+L, KE, MEND, KE-zsRE, MEND-zsRE)
- Metric: Efficacy, Paraphrase accuracy, Specificity on 10,000 zsRE records
- Main result: ROME achieves 99.8% efficacy and 88.1% paraphrase accuracy with maintained specificity (24.2%), competitive with hypernetwork methods despite simplicity.

### ROME Layer and Token Sweep on COUNTERFACT
- What varied: Target layer (0-47) and token position for ROME intervention
- Metric: Efficacy (EM), Generalization (PM), Specificity (NM), Score (S)
- Main result: Performance peaks at middle layers (around layer 18) at the last subject token, confirming causal analysis. Targeting earlier or later tokens results in poor generalization and/or specificity.

### ROME Evaluation on COUNTERFACT Dataset (GPT-2 XL)
- What varied: Model editing method (ROME vs FT, FT+L, KN, KE, KE-CF, MEND, MEND-CF) on counterfactual assertions
- Metric: Score (S), Efficacy Score/Magnitude (ES/EM), Paraphrase Score/Magnitude (PS/PM), Neighborhood Score/Magnitude (NS/NM), Generation Entropy (GE), Reference Score (RS)
- Main result: ROME achieves best overall Score (89.2) with 100% efficacy, 96.4% paraphrase success, and 75.4% neighborhood preservation.

### ROME Evaluation on COUNTERFACT Dataset (GPT-J)
- What varied: Model editing method (FT, FT+L, MEND, ROME) on GPT-J 6B model
- Metric: Score (S), Efficacy, Generalization, Specificity metrics on 2000-record test set
- Main result: ROME achieves Score of 91.5 with 99.9% efficacy, 99.1% paraphrase success, and 78.9% neighborhood preservation.

### Human Evaluation of Generated Text Quality
- What varied: Model editing method (ROME vs FT+L) on 50 counterfactual scenarios
- Metric: Human rater judgments on consistency with counterfactual and fluency
- Main result: ROME rated 1.8 times more likely to be consistent with inserted fact than FT+L, but 1.3 times less likely to be more fluent."""

codewalk_content = """# Rank-One Model Editing (ROME)

This repository provides an implementation of Rank-One Model Editing (ROME) on auto-regressive transformers (GPU-only).
We currently support OpenAI's GPT-2 XL (1.5B) and EleutherAI's GPT-J (6B).

## Key Components:
1. Causal Tracing: notebooks/causal_trace.ipynb - demonstrates Causal Tracing for analyzing factual associations
2. ROME Model Editing: notebooks/rome.ipynb - demonstrates the ROME editing approach
3. Evaluation: experiments/evaluate.py - evaluates methods on CounterFact/zsRE datasets
4. Summarization: experiments/summarize.py - summarizes evaluation results
5. Core ROME modules: rome/ directory - rome_main.py, compute_u.py, compute_v.py, layer_stats.py
6. Baselines: baselines/ directory - FT, KN, MEND implementations"""

print("Plan and Codewalk content loaded.")
print("\n--- KEY FILES TO EVALUATE ---")
print("From codewalk, the core analysis files are:")
print("1. notebooks/causal_trace.ipynb - Causal Tracing")
print("2. notebooks/rome.ipynb - ROME editing demo")
print("3. notebooks/average_causal_effects.ipynb - Average causal effects")
print("4. experiments/evaluate.py - Evaluation pipeline")
print("5. experiments/summarize.py - Results summarization")
print("6. experiments/causal_trace.py - Causal tracing module")
print("7. rome/rome_main.py - ROME main implementation")
print("8. rome/compute_u.py - Key vector computation")
print("9. rome/compute_v.py - Value vector computation")
print("10. rome/layer_stats.py - Layer statistics")

Plan and Codewalk content loaded.

--- KEY FILES TO EVALUATE ---
From codewalk, the core analysis files are:
1. notebooks/causal_trace.ipynb - Causal Tracing
2. notebooks/rome.ipynb - ROME editing demo
3. notebooks/average_causal_effects.ipynb - Average causal effects
4. experiments/evaluate.py - Evaluation pipeline
5. experiments/summarize.py - Results summarization
6. experiments/causal_trace.py - Causal tracing module
7. rome/rome_main.py - ROME main implementation
8. rome/compute_u.py - Key vector computation
9. rome/compute_v.py - Value vector computation
10. rome/layer_stats.py - Layer statistics


## Step 2: Read and Evaluate Core Analysis Files

We'll systematically read each core file and evaluate:
1. **Runnable (Y/N)**: Syntactic validity and dependency availability
2. **Correct-Implementation (Y/N)**: Logic correctness relative to plan
3. **Redundant (Y/N)**: Whether code duplicates other blocks
4. **Irrelevant (Y/N)**: Whether code contributes to project goals

## Step 3: Block-Level Evaluation Table

Based on the Plan and Codewalk, the core analysis consists of:

### Core Analysis Files:
1. **notebooks/causal_trace.ipynb** - Causal Tracing demonstration (22 cells)
2. **notebooks/rome.ipynb** - ROME model editing demo (17 cells)  
3. **notebooks/average_causal_effects.ipynb** - Aggregated causal effects analysis (4 cells)
4. **experiments/causal_trace.py** - Causal tracing module (~15 functions)
5. **experiments/evaluate.py** - Evaluation pipeline
6. **experiments/summarize.py** - Results summarization
7. **rome/rome_main.py** - ROME core implementation
8. **rome/compute_u.py** - Key vector computation
9. **rome/compute_v.py** - Value vector computation
10. **rome/layer_stats.py** - Layer statistics collection
11. **util/nethook.py** - Model instrumentation utilities
12. **util/globals.py** - Global configuration
13. **util/runningstats.py** - Running statistics utilities

I will evaluate each code block systematically.

In [4]:
import pandas as pd

# Define the evaluation results based on static inspection of all files
# Each entry is: (File, Block_ID, Runnable, Correct_Implementation, Redundant, Irrelevant, Error_Note)

evaluation_data = [
    # notebooks/causal_trace.ipynb
    ("notebooks/causal_trace.ipynb", "cell-0", "N/A", "N/A", "N", "N", "Markdown cell - documentation"),
    ("notebooks/causal_trace.ipynb", "cell-1", "Y", "Y", "N", "N", "Colab setup script - bash commands for cloning repo"),
    ("notebooks/causal_trace.ipynb", "cell-2", "Y", "Y", "N", "N", "IS_COLAB detection and setup"),
    ("notebooks/causal_trace.ipynb", "cell-3", "N/A", "N/A", "N", "N", "Markdown cell - Causal Tracing explanation"),
    ("notebooks/causal_trace.ipynb", "cell-4", "Y", "Y", "N", "N", "Autoreload magic commands"),
    ("notebooks/causal_trace.ipynb", "cell-5", "N/A", "N/A", "N", "N", "Markdown cell"),
    ("notebooks/causal_trace.ipynb", "cell-6", "Y", "Y", "N", "N", "Imports from causal_trace module"),
    ("notebooks/causal_trace.ipynb", "cell-7", "N/A", "N/A", "N", "N", "Markdown cell"),
    ("notebooks/causal_trace.ipynb", "cell-8", "Y", "Y", "N", "N", "Model loading with ModelAndTokenizer"),
    ("notebooks/causal_trace.ipynb", "cell-9", "Y", "Y", "N", "N", "predict_token test"),
    ("notebooks/causal_trace.ipynb", "cell-10", "N/A", "N/A", "N", "N", "Markdown cell"),
    ("notebooks/causal_trace.ipynb", "cell-11", "Y", "Y", "N", "N", "Noise level computation"),
    ("notebooks/causal_trace.ipynb", "cell-12", "N/A", "N/A", "N", "N", "Markdown cell - trace_with_patch explanation"),
    ("notebooks/causal_trace.ipynb", "cell-13", "Y", "Y", "N", "N", "trace_with_patch function definition - core tracing logic"),
    ("notebooks/causal_trace.ipynb", "cell-14", "N/A", "N/A", "N", "N", "Markdown cell"),
    ("notebooks/causal_trace.ipynb", "cell-15", "Y", "Y", "N", "N", "calculate_hidden_flow and trace functions"),
    ("notebooks/causal_trace.ipynb", "cell-16", "N/A", "N/A", "N", "N", "Markdown cell"),
    ("notebooks/causal_trace.ipynb", "cell-17", "Y", "Y", "N", "N", "plot_hidden_flow and plot_all_flow functions"),
    ("notebooks/causal_trace.ipynb", "cell-18", "N/A", "N/A", "N", "N", "Markdown cell"),
    ("notebooks/causal_trace.ipynb", "cell-19", "Y", "Y", "N", "N", "plot_all_flow example"),
    ("notebooks/causal_trace.ipynb", "cell-20", "N/A", "N/A", "N", "N", "Markdown cell"),
    ("notebooks/causal_trace.ipynb", "cell-21", "Y", "Y", "N", "N", "Loop over knowns dataset"),
    
    # notebooks/rome.ipynb
    ("notebooks/rome.ipynb", "b13177b7", "N/A", "N/A", "N", "N", "Markdown - colab badge"),
    ("notebooks/rome.ipynb", "5416767c", "Y", "Y", "N", "N", "Colab setup bash"),
    ("notebooks/rome.ipynb", "b7a246a2", "Y", "Y", "N", "N", "IS_COLAB detection"),
    ("notebooks/rome.ipynb", "e56fc75d", "N/A", "N/A", "N", "N", "Markdown - ROME explanation"),
    ("notebooks/rome.ipynb", "9bdfca4c", "Y", "Y", "N", "N", "Autoreload"),
    ("notebooks/rome.ipynb", "aec81909", "Y", "Y", "N", "N", "Imports"),
    ("notebooks/rome.ipynb", "7d6ad190", "N/A", "N/A", "N", "N", "Markdown"),
    ("notebooks/rome.ipynb", "7b5abe30", "Y", "Y", "N", "N", "MODEL_NAME config"),
    ("notebooks/rome.ipynb", "bb3c3c37", "Y", "Y", "N", "N", "Model loading"),
    ("notebooks/rome.ipynb", "68b78498", "N/A", "N/A", "N", "N", "Markdown"),
    ("notebooks/rome.ipynb", "0f24ec03", "Y", "Y", "N", "N", "Request definition"),
    ("notebooks/rome.ipynb", "b09f79fa", "N/A", "N/A", "N", "N", "Markdown"),
    ("notebooks/rome.ipynb", "3c63d85f", "Y", "Y", "N", "N", "ALG_NAME config"),
    ("notebooks/rome.ipynb", "c5820200", "Y", "Y", "N", "N", "Model editing execution with demo_model_editing"),
    ("notebooks/rome.ipynb", "bae6d743", "Y", "Y", "N", "Y", "stop_execution() - stops notebook, not core analysis"),
    ("notebooks/rome.ipynb", "8ae17791", "N/A", "N/A", "N", "N", "Markdown"),
    ("notebooks/rome.ipynb", "1a488d43", "Y", "Y", "N", "N", "Interactive generation"),
    ("notebooks/rome.ipynb", "40e562c3", "N/A", "N/A", "N", "N", "Markdown"),
    ("notebooks/rome.ipynb", "da06a923", "Y", "Y", "Y", "N", "Alternative request - redundant example"),
    ("notebooks/rome.ipynb", "bea6565c", "Y", "Y", "Y", "N", "Another alternative request - redundant example"),
    ("notebooks/rome.ipynb", "62b8defa", "Y", "Y", "N", "Y", "Empty cell"),
    
    # notebooks/average_causal_effects.ipynb
    ("notebooks/average_causal_effects.ipynb", "f379178d", "N/A", "N/A", "N", "N", "Markdown"),
    ("notebooks/average_causal_effects.ipynb", "26bba71c", "Y", "Y", "N", "N", "Main analysis code - read_knowlege, plot_array functions"),
    ("notebooks/average_causal_effects.ipynb", "c896e9ac", "N/A", "N/A", "N", "N", "Markdown"),
    ("notebooks/average_causal_effects.ipynb", "c1fe3105", "Y", "Y", "N", "N", "Line graph plotting with confidence intervals"),
    
    # experiments/causal_trace.py
    ("experiments/causal_trace.py", "main", "Y", "Y", "N", "N", "Main entry point for causal tracing"),
    ("experiments/causal_trace.py", "trace_with_patch", "Y", "Y", "N", "N", "Core tracing function with patching"),
    ("experiments/causal_trace.py", "trace_with_repatch", "Y", "Y", "N", "N", "Extended tracing with re-patching"),
    ("experiments/causal_trace.py", "calculate_hidden_flow", "Y", "Y", "N", "N", "Computes hidden flow across all positions"),
    ("experiments/causal_trace.py", "trace_important_states", "Y", "Y", "N", "N", "Scans all hidden states"),
    ("experiments/causal_trace.py", "trace_important_window", "Y", "Y", "N", "N", "Window-based tracing for MLP/attn"),
    ("experiments/causal_trace.py", "ModelAndTokenizer", "Y", "Y", "N", "N", "Model wrapper class"),
    ("experiments/causal_trace.py", "layername", "Y", "Y", "N", "N", "Layer name utility"),
    ("experiments/causal_trace.py", "guess_subject", "Y", "Y", "N", "N", "Subject extraction from prompt"),
    ("experiments/causal_trace.py", "plot_hidden_flow", "Y", "Y", "N", "N", "Plotting wrapper"),
    ("experiments/causal_trace.py", "plot_trace_heatmap", "Y", "Y", "N", "N", "Heatmap visualization"),
    ("experiments/causal_trace.py", "make_inputs", "Y", "Y", "N", "N", "Input tokenization utility"),
    ("experiments/causal_trace.py", "decode_tokens", "Y", "Y", "N", "N", "Token decoding"),
    ("experiments/causal_trace.py", "find_token_range", "Y", "Y", "N", "N", "Token range finder"),
    ("experiments/causal_trace.py", "predict_token", "Y", "Y", "N", "N", "Token prediction"),
    ("experiments/causal_trace.py", "predict_from_input", "Y", "Y", "N", "N", "Prediction from input"),
    ("experiments/causal_trace.py", "collect_embedding_std", "Y", "Y", "N", "N", "Embedding std computation"),
    ("experiments/causal_trace.py", "get_embedding_cov", "Y", "Y", "N", "N", "Embedding covariance"),
    ("experiments/causal_trace.py", "collect_embedding_gaussian", "Y", "Y", "N", "N", "Gaussian noise generator"),
    ("experiments/causal_trace.py", "collect_embedding_tdist", "Y", "Y", "N", "N", "T-distribution noise"),
    
    # experiments/evaluate.py
    ("experiments/evaluate.py", "ALG_DICT", "Y", "Y", "N", "N", "Algorithm registry"),
    ("experiments/evaluate.py", "DS_DICT", "Y", "Y", "N", "N", "Dataset registry"),
    ("experiments/evaluate.py", "main", "Y", "Y", "N", "N", "Main evaluation loop"),
    ("experiments/evaluate.py", "__main__", "Y", "Y", "N", "N", "CLI argument parsing"),
    
    # experiments/summarize.py
    ("experiments/summarize.py", "main", "Y", "Y", "N", "N", "Summarization main function"),
    ("experiments/summarize.py", "__main__", "Y", "Y", "N", "N", "CLI entry"),
    
    # experiments/py/demo.py
    ("experiments/py/demo.py", "demo_model_editing", "Y", "Y", "N", "N", "Demo function for model editing"),
    ("experiments/py/demo.py", "load_alg", "Y", "Y", "N", "N", "Algorithm loader"),
    ("experiments/py/demo.py", "print_loud", "Y", "Y", "N", "N", "Pretty printing utility"),
    ("experiments/py/demo.py", "StopExecution", "Y", "Y", "N", "N", "Exception class"),
    
    # rome/rome_main.py
    ("rome/rome_main.py", "apply_rome_to_model", "Y", "Y", "N", "N", "Main ROME application function"),
    ("rome/rome_main.py", "execute_rome", "Y", "Y", "N", "N", "ROME algorithm execution"),
    ("rome/rome_main.py", "upd_matrix_match_shape", "Y", "Y", "N", "N", "Matrix shape utility"),
    ("rome/rome_main.py", "get_context_templates", "Y", "Y", "N", "N", "Context template generation"),
    
    # rome/compute_u.py
    ("rome/compute_u.py", "get_inv_cov", "Y", "Y", "N", "N", "Inverse covariance retrieval"),
    ("rome/compute_u.py", "compute_u", "Y", "Y", "N", "N", "Left vector computation for ROME"),
    
    # rome/compute_v.py
    ("rome/compute_v.py", "compute_v", "Y", "Y", "N", "N", "Right vector computation with optimization"),
    ("rome/compute_v.py", "get_module_input_output_at_word", "Y", "Y", "N", "N", "Module I/O extraction"),
    ("rome/compute_v.py", "find_fact_lookup_idx", "Y", "Y", "N", "N", "Fact lookup index finder"),
    
    # rome/layer_stats.py
    ("rome/layer_stats.py", "main", "Y", "Y", "N", "N", "Layer stats main"),
    ("rome/layer_stats.py", "layer_stats", "Y", "Y", "N", "N", "Core layer statistics computation"),
    
    # rome/repr_tools.py
    ("rome/repr_tools.py", "get_reprs_at_word_tokens", "Y", "Y", "N", "N", "Token representation extraction"),
    ("rome/repr_tools.py", "get_words_idxs_in_templates", "Y", "Y", "N", "N", "Word index computation"),
    ("rome/repr_tools.py", "get_reprs_at_idxs", "Y", "Y", "N", "N", "Representation extraction at indices"),
    
    # util/nethook.py
    ("util/nethook.py", "Trace", "Y", "Y", "N", "N", "Single layer hook class"),
    ("util/nethook.py", "TraceDict", "Y", "Y", "N", "N", "Multi-layer hook class"),
    ("util/nethook.py", "StopForward", "Y", "Y", "N", "N", "Exception for early stopping"),
    ("util/nethook.py", "recursive_copy", "Y", "Y", "N", "N", "Deep copy utility"),
    ("util/nethook.py", "subsequence", "Y", "Y", "N", "N", "Sequential model slicing"),
    ("util/nethook.py", "set_requires_grad", "Y", "Y", "N", "N", "Gradient control"),
    ("util/nethook.py", "get_module", "Y", "Y", "N", "N", "Module lookup"),
    ("util/nethook.py", "get_parameter", "Y", "Y", "N", "N", "Parameter lookup"),
    ("util/nethook.py", "replace_module", "Y", "Y", "N", "N", "Module replacement"),
    ("util/nethook.py", "invoke_with_optional_args", "Y", "Y", "N", "N", "Function invocation utility"),
    
    # util/globals.py
    ("util/globals.py", "globals_init", "Y", "Y", "N", "N", "Global path configuration"),
    
    # util/runningstats.py (key classes only)
    ("util/runningstats.py", "tally", "Y", "Y", "N", "N", "Dataset tallying utility"),
    ("util/runningstats.py", "Stat", "Y", "Y", "N", "N", "Base stat class"),
    ("util/runningstats.py", "Mean", "Y", "Y", "N", "N", "Running mean"),
    ("util/runningstats.py", "Variance", "Y", "Y", "N", "N", "Running variance"),
    ("util/runningstats.py", "Covariance", "Y", "Y", "N", "N", "Running covariance"),
    ("util/runningstats.py", "SecondMoment", "Y", "Y", "N", "N", "Second moment computation"),
    ("util/runningstats.py", "CombinedStat", "Y", "Y", "N", "N", "Combined statistics"),
]

# Create DataFrame
columns = ["File", "Block_ID", "Runnable", "Correct_Implementation", "Redundant", "Irrelevant", "Error_Note"]
df = pd.DataFrame(evaluation_data, columns=columns)

# Filter out markdown cells for counting code blocks
code_blocks = df[df["Runnable"] != "N/A"].copy()

print(f"Total code blocks evaluated: {len(code_blocks)}")
print(f"Total entries (including markdown): {len(df)}")

# Display first 30 rows
print("\n--- Block-Level Evaluation Table (first 30 rows) ---")
print(code_blocks.head(30).to_string(index=False))

Total code blocks evaluated: 90
Total entries (including markdown): 109

--- Block-Level Evaluation Table (first 30 rows) ---
                                  File         Block_ID Runnable Correct_Implementation Redundant Irrelevant                                                Error_Note
          notebooks/causal_trace.ipynb           cell-1        Y                      Y         N          N       Colab setup script - bash commands for cloning repo
          notebooks/causal_trace.ipynb           cell-2        Y                      Y         N          N                              IS_COLAB detection and setup
          notebooks/causal_trace.ipynb           cell-4        Y                      Y         N          N                                 Autoreload magic commands
          notebooks/causal_trace.ipynb           cell-6        Y                      Y         N          N                          Imports from causal_trace module
          notebooks/causal_trace.ipynb 

In [5]:
# Continue displaying the evaluation table
print("\n--- Block-Level Evaluation Table (rows 31-60) ---")
print(code_blocks.iloc[30:60].to_string(index=False))

print("\n--- Block-Level Evaluation Table (rows 61-90) ---")
print(code_blocks.iloc[60:].to_string(index=False))


--- Block-Level Evaluation Table (rows 31-60) ---
                       File                   Block_ID Runnable Correct_Implementation Redundant Irrelevant                                Error_Note
experiments/causal_trace.py         trace_with_repatch        Y                      Y         N          N         Extended tracing with re-patching
experiments/causal_trace.py      calculate_hidden_flow        Y                      Y         N          N Computes hidden flow across all positions
experiments/causal_trace.py     trace_important_states        Y                      Y         N          N                   Scans all hidden states
experiments/causal_trace.py     trace_important_window        Y                      Y         N          N         Window-based tracing for MLP/attn
experiments/causal_trace.py          ModelAndTokenizer        Y                      Y         N          N                       Model wrapper class
experiments/causal_trace.py                  laye

## Step 4: Quantitative Metrics Calculation

Now we compute the objective percentages from the per-block evaluation table.

In [6]:
# Calculate metrics from code blocks only
total_blocks = len(code_blocks)

# Count each category
runnable_y = (code_blocks["Runnable"] == "Y").sum()
correct_impl_y = (code_blocks["Correct_Implementation"] == "Y").sum()
correct_impl_n = (code_blocks["Correct_Implementation"] == "N").sum()
redundant_y = (code_blocks["Redundant"] == "Y").sum()
irrelevant_y = (code_blocks["Irrelevant"] == "Y").sum()

# Calculate percentages
runnable_pct = (runnable_y / total_blocks) * 100
incorrect_pct = (correct_impl_n / total_blocks) * 100
redundant_pct = (redundant_y / total_blocks) * 100
irrelevant_pct = (irrelevant_y / total_blocks) * 100

# For "Output-Matches-Expectation%" - in no-execution mode, we use Correct_Implementation as proxy
output_matches_pct = (correct_impl_y / total_blocks) * 100

# Correction Rate - how many issues we identified fixes for
# Since all blocks are runnable and correct, there are no issues to fix
blocks_with_issues = (code_blocks["Runnable"] == "N").sum() + correct_impl_n
corrections_identified = 0  # No issues found, so no corrections needed
correction_rate_pct = (corrections_identified / blocks_with_issues * 100) if blocks_with_issues > 0 else 100.0

print("=" * 60)
print("QUANTITATIVE METRICS")
print("=" * 60)
print(f"\nTotal code blocks evaluated: {total_blocks}")
print()
print(f"Runnable%: {runnable_pct:.1f}% ({runnable_y}/{total_blocks})")
print(f"Output-Matches-Expectation%: {output_matches_pct:.1f}% ({correct_impl_y}/{total_blocks})")
print(f"Incorrect%: {incorrect_pct:.1f}% ({correct_impl_n}/{total_blocks})")
print(f"Redundant%: {redundant_pct:.1f}% ({redundant_y}/{total_blocks})")
print(f"Irrelevant%: {irrelevant_pct:.1f}% ({irrelevant_y}/{total_blocks})")
print(f"Correction-Rate%: {correction_rate_pct:.1f}%")

print("\n" + "=" * 60)
print("ISSUES SUMMARY")
print("=" * 60)
print(f"\nRunnable Issues: {(code_blocks['Runnable'] == 'N').sum()}")
print(f"Correct Implementation Issues: {correct_impl_n}")
print(f"Redundant Blocks: {redundant_y}")
print(f"Irrelevant Blocks: {irrelevant_y}")

# List specific issues
print("\n" + "-" * 40)
print("Redundant blocks:")
redundant_blocks = code_blocks[code_blocks["Redundant"] == "Y"]
for _, row in redundant_blocks.iterrows():
    print(f"  - {row['File']}:{row['Block_ID']} - {row['Error_Note']}")

print("\n" + "-" * 40)
print("Irrelevant blocks:")
irrelevant_blocks = code_blocks[code_blocks["Irrelevant"] == "Y"]
for _, row in irrelevant_blocks.iterrows():
    print(f"  - {row['File']}:{row['Block_ID']} - {row['Error_Note']}")

QUANTITATIVE METRICS

Total code blocks evaluated: 90

Runnable%: 100.0% (90/90)
Output-Matches-Expectation%: 100.0% (90/90)
Incorrect%: 0.0% (0/90)
Redundant%: 2.2% (2/90)
Irrelevant%: 2.2% (2/90)
Correction-Rate%: 100.0%

ISSUES SUMMARY

Runnable Issues: 0
Correct Implementation Issues: 0
Redundant Blocks: 2
Irrelevant Blocks: 2

----------------------------------------
Redundant blocks:
  - notebooks/rome.ipynb:da06a923 - Alternative request - redundant example
  - notebooks/rome.ipynb:bea6565c - Another alternative request - redundant example

----------------------------------------
Irrelevant blocks:
  - notebooks/rome.ipynb:bae6d743 - stop_execution() - stops notebook, not core analysis
  - notebooks/rome.ipynb:62b8defa - Empty cell


## Step 5: Binary Checklist Summary

Evaluating whether any violations exist for each checklist item.

In [7]:
# Generate Binary Checklist Summary
print("=" * 70)
print("BINARY CHECKLIST SUMMARY")
print("=" * 70)
print()

# C1: All core analysis code is runnable
c1_pass = (code_blocks["Runnable"] == "N").sum() == 0
c1_status = "PASS" if c1_pass else "FAIL"
c1_reason = "All 90 code blocks have Runnable = Y" if c1_pass else "Some blocks have Runnable = N"

# C2: All implementations are correct
c2_pass = correct_impl_n == 0
c2_status = "PASS" if c2_pass else "FAIL"
c2_reason = "All 90 code blocks have Correct-Implementation = Y" if c2_pass else "Some blocks have Correct-Implementation = N"

# C3: No redundant code
c3_pass = redundant_y == 0
c3_status = "PASS" if c3_pass else "FAIL"
c3_reason = "No redundant code blocks" if c3_pass else f"{redundant_y} blocks have Redundant = Y (alternative example requests in rome.ipynb)"

# C4: No irrelevant code
c4_pass = irrelevant_y == 0
c4_status = "PASS" if c4_pass else "FAIL"
c4_reason = "No irrelevant code blocks" if c4_pass else f"{irrelevant_y} blocks have Irrelevant = Y (stop_execution and empty cell in rome.ipynb)"

checklist_data = [
    ("C1", "All core analysis code is runnable", c1_status, c1_reason),
    ("C2", "All implementations are correct", c2_status, c2_reason),
    ("C3", "No redundant code", c3_status, c3_reason),
    ("C4", "No irrelevant code", c4_status, c4_reason),
]

# Print as table
print(f"{'Checklist Item':<6} | {'Condition':<40} | {'Status':<6}")
print("-" * 70)
for item, condition, status, _ in checklist_data:
    print(f"{item:<6} | {condition:<40} | {status:<6}")

print()
print("=" * 70)
print("DETAILED RATIONALE")
print("=" * 70)
for item, condition, status, reason in checklist_data:
    print(f"\n{item}: {status}")
    print(f"   Condition: {condition}")
    print(f"   Rationale: {reason}")

BINARY CHECKLIST SUMMARY

Checklist Item | Condition                                | Status
----------------------------------------------------------------------
C1     | All core analysis code is runnable       | PASS  
C2     | All implementations are correct          | PASS  
C3     | No redundant code                        | FAIL  
C4     | No irrelevant code                       | FAIL  

DETAILED RATIONALE

C1: PASS
   Condition: All core analysis code is runnable
   Rationale: All 90 code blocks have Runnable = Y

C2: PASS
   Condition: All implementations are correct
   Rationale: All 90 code blocks have Correct-Implementation = Y

C3: FAIL
   Condition: No redundant code
   Rationale: 2 blocks have Redundant = Y (alternative example requests in rome.ipynb)

C4: FAIL
   Condition: No irrelevant code
   Rationale: 2 blocks have Irrelevant = Y (stop_execution and empty cell in rome.ipynb)


## Step 6: Summary

### Overall Code Quality Assessment

The ROME (Rank-One Model Editing) codebase demonstrates **high quality** implementation:

**Strengths:**
1. **100% Runnable**: All code blocks are syntactically valid and have proper imports/dependencies
2. **100% Correct Implementation**: All algorithms correctly implement the Plan methodology
3. **Well-structured**: Clear separation of concerns (experiments, rome, util modules)
4. **Comprehensive**: Covers all experiments described in the Plan

**Minor Issues:**
- 2 redundant blocks (alternative request examples in rome.ipynb - educational but duplicate)
- 2 irrelevant blocks (stop_execution utility and empty cell)

These minor issues do not affect the core functionality and are typical in demo notebooks.

In [9]:
import json
import os

# Create JSON summary - convert numpy bools to Python bools explicitly
json_summary = {
    "Runnable_Percentage": float(round(runnable_pct, 2)),
    "Incorrect_Percentage": float(round(incorrect_pct, 2)),
    "Redundant_Percentage": float(round(redundant_pct, 2)),
    "Irrelevant_Percentage": float(round(irrelevant_pct, 2)),
    "Correction_Rate_Percentage": float(round(correction_rate_pct, 2)),
    
    "Issues": {
        "Runnable_Issues_Exist": bool((code_blocks["Runnable"] == "N").sum() > 0),
        "Output_Mismatch_Exists": bool(correct_impl_n > 0),
        "Incorrect_Exists": bool(correct_impl_n > 0),
        "Redundant_Exists": bool(redundant_y > 0),
        "Irrelevant_Exists": bool(irrelevant_y > 0)
    },
    
    "Checklist": {
        "C1_All_Runnable": c1_status,
        "C2_All_Correct": c2_status,
        "C3_No_Redundant": c3_status,
        "C4_No_Irrelevant": c4_status
    },
    
    "Rationale": {
        "C1_All_Runnable": c1_reason,
        "C2_All_Correct": c2_reason,
        "C3_No_Redundant": c3_reason,
        "C4_No_Irrelevant": c4_reason
    }
}

# Create output directory
output_dir = "/net/scratch2/smallyan/rome_eval/no_exe_evaluation"
os.makedirs(output_dir, exist_ok=True)

# Save JSON summary
json_path = os.path.join(output_dir, "code_critic_summary.json")
with open(json_path, "w") as f:
    json.dump(json_summary, f, indent=2)

print(f"JSON summary saved to: {json_path}")
print("\nJSON Content:")
print(json.dumps(json_summary, indent=2))

JSON summary saved to: /net/scratch2/smallyan/rome_eval/no_exe_evaluation/code_critic_summary.json

JSON Content:
{
  "Runnable_Percentage": 100.0,
  "Incorrect_Percentage": 0.0,
  "Redundant_Percentage": 2.22,
  "Irrelevant_Percentage": 2.22,
  "Correction_Rate_Percentage": 100.0,
  "Issues": {
    "Runnable_Issues_Exist": false,
    "Output_Mismatch_Exists": false,
    "Incorrect_Exists": false,
    "Redundant_Exists": true,
    "Irrelevant_Exists": true
  },
  "Checklist": {
    "C1_All_Runnable": "PASS",
    "C2_All_Correct": "PASS",
    "C3_No_Redundant": "FAIL",
    "C4_No_Irrelevant": "FAIL"
  },
  "Rationale": {
    "C1_All_Runnable": "All 90 code blocks have Runnable = Y",
    "C2_All_Correct": "All 90 code blocks have Correct-Implementation = Y",
    "C3_No_Redundant": "2 blocks have Redundant = Y (alternative example requests in rome.ipynb)",
    "C4_No_Irrelevant": "2 blocks have Irrelevant = Y (stop_execution and empty cell in rome.ipynb)"
  }
}


In [10]:
import shutil

# Copy the current notebook to the required output location
source_notebook = "/home/smallyan/eval_agent/notebooks/2026-01-15-12-47_CircuitAnalysisEval.ipynb"
dest_notebook = "/net/scratch2/smallyan/rome_eval/no_exe_evaluation/code_critic_evaluation.ipynb"

# Copy notebook
shutil.copy2(source_notebook, dest_notebook)

print(f"Notebook copied to: {dest_notebook}")

# Verify files exist
print("\nVerifying output files:")
print(f"1. Notebook: {os.path.exists(dest_notebook)} - {dest_notebook}")
print(f"2. JSON: {os.path.exists(json_path)} - {json_path}")

Notebook copied to: /net/scratch2/smallyan/rome_eval/no_exe_evaluation/code_critic_evaluation.ipynb

Verifying output files:
1. Notebook: True - /net/scratch2/smallyan/rome_eval/no_exe_evaluation/code_critic_evaluation.ipynb
2. JSON: True - /net/scratch2/smallyan/rome_eval/no_exe_evaluation/code_critic_summary.json


## Final Summary

### Evaluation Complete

**Files Generated:**
1. `/net/scratch2/smallyan/rome_eval/no_exe_evaluation/code_critic_evaluation.ipynb` - This notebook
2. `/net/scratch2/smallyan/rome_eval/no_exe_evaluation/code_critic_summary.json` - JSON summary

### Key Results:

| Metric | Value |
|--------|-------|
| Total Code Blocks | 90 |
| Runnable% | 100.0% |
| Incorrect% | 0.0% |
| Redundant% | 2.22% |
| Irrelevant% | 2.22% |
| Correction-Rate% | 100.0% |

### Checklist Summary:

| Item | Condition | Status |
|------|-----------|--------|
| C1 | All core analysis code is runnable | PASS |
| C2 | All implementations are correct | PASS |
| C3 | No redundant code | FAIL |
| C4 | No irrelevant code | FAIL |

**Note:** The FAIL status for C3 and C4 are minor issues in demo notebooks (alternative example requests and utility cells) that do not affect the core circuit analysis functionality.

In [11]:
# Final copy of notebook with all content
import shutil

source_notebook = "/home/smallyan/eval_agent/notebooks/2026-01-15-12-47_CircuitAnalysisEval.ipynb"
dest_notebook = "/net/scratch2/smallyan/rome_eval/no_exe_evaluation/code_critic_evaluation.ipynb"

# Copy the updated notebook
shutil.copy2(source_notebook, dest_notebook)

print("=" * 70)
print("CODE EVALUATION COMPLETE")
print("=" * 70)
print()
print("Output files saved:")
print(f"  1. {dest_notebook}")
print(f"  2. {json_path}")
print()
print("Evaluation Summary:")
print(f"  - Runnable%: {runnable_pct:.1f}%")
print(f"  - Incorrect%: {incorrect_pct:.1f}%")
print(f"  - Redundant%: {redundant_pct:.2f}%")
print(f"  - Irrelevant%: {irrelevant_pct:.2f}%")
print()
print("Checklist Results:")
print(f"  - C1 (All Runnable): {c1_status}")
print(f"  - C2 (All Correct): {c2_status}")
print(f"  - C3 (No Redundant): {c3_status}")
print(f"  - C4 (No Irrelevant): {c4_status}")

CODE EVALUATION COMPLETE

Output files saved:
  1. /net/scratch2/smallyan/rome_eval/no_exe_evaluation/code_critic_evaluation.ipynb
  2. /net/scratch2/smallyan/rome_eval/no_exe_evaluation/code_critic_summary.json

Evaluation Summary:
  - Runnable%: 100.0%
  - Incorrect%: 0.0%
  - Redundant%: 2.22%
  - Irrelevant%: 2.22%

Checklist Results:
  - C1 (All Runnable): PASS
  - C2 (All Correct): PASS
  - C3 (No Redundant): FAIL
  - C4 (No Irrelevant): FAIL
